# Aula 05 — treinamento de cabeças, versão 02

**Objetivo:** transformar textos em vetores com o Phi-4 congelado, treinar duas cabeças e avaliar suas saídas em dados reservados.

Execute as células na ordem. Na seção 2, o pacote **dados-colab-v02.zip** é baixado do GitHub. O pacote contém entradas, supervisão, instruções e referências para comparação; não contém o corpo Phi-4, que será baixado na seção 5.

```text
Texto autorizado → Phi-4 congelado → vetor de 5.120 valores → cabeça treinável
                                                          ├─ 3 classes
                                                          └─ probabilidade de evento
```

O alvo de classificação é a relação entre uma afirmação e suas evidências. O alvo probabilístico é a frequência de um evento sob condições explícitas de continuação. **Não há alvo de ação, escolha de estratégia ou criação de plano.**

Esta é uma nova execução: dados v05, treinamento v02. O notebook, os dados e as cabeças anteriores ficam preservados. A extração dos vetores é a etapa mais demorada; o treinamento das pequenas cabeças vem depois.

## 1. Preparar o ambiente

Instalamos as versões usadas pelo carregador portátil. O PyTorch fornecido pelo Colab será utilizado. O corpo escolhe automaticamente CUDA, MPS ou CPU; memória e tempo de extração dependem da máquina.

In [ ]:
!pip install -q transformers==5.0.0 safetensors==0.8.0 huggingface-hub==1.23.0

## 2. Carregar o pacote da nova versão

Baixamos **dados-colab-v02.zip** da `main` do GitHub. Entradas e alvos estão em arquivos separados. A auditoria privada da engine não faz parte deste pacote de treinamento.

`WORK_DIR` guarda os vetores e pesos desta execução. Para retomar após perder a sessão, use uma pasta persistente, por exemplo no Drive já montado, e execute novamente a célula de carregamento.

In [ ]:
from pathlib import Path
from google.colab import files
import gzip, json, hashlib, zipfile, io, time
from collections import Counter
import numpy as np
import pandas as pd
import torch
from IPython.display import display

WORK_DIR = Path("/content/treinamento-aula05-v02")
WORK_DIR.mkdir(parents=True, exist_ok=True)
from urllib.request import urlopen
URL_DADOS = "https://raw.githubusercontent.com/AIConnection/missao-espacial/main/treinamento/piloto-v02/dados-colab-v02.zip?formato=agent-view-compact"
with urlopen(URL_DADOS) as resposta:
    dados_zip = resposta.read()
with zipfile.ZipFile(io.BytesIO(dados_zip)) as pacote:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in pacote.namelist())
    pacote.extractall(WORK_DIR / "pacote")
DATA_DIR = WORK_DIR / "pacote"
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()
lock = json.loads((DATA_DIR / "integridade.json").read_text())
assert all(sha(DATA_DIR / nome) == valor for nome, valor in lock.items()), "Pacote alterado"
with gzip.open(DATA_DIR / "inputs.jsonl.gz", "rt", encoding="utf-8") as f:
    inputs = [json.loads(l) for l in f]
supervision = [json.loads(l) for l in (DATA_DIR / "supervisao.jsonl").read_text().splitlines()]
momentos = json.loads((DATA_DIR / "momentos.json").read_text())
for r in supervision:
    if r["task"] == "forecast":
        r.update(momentos[r["root"]])
manifesto = json.loads((DATA_DIR / "manifesto.json").read_text())
protocol = json.loads((DATA_DIR / "protocolo.json").read_text())
CLASSES = ["suporte", "insuficiente", "contradição"]
assert len(inputs) == len(supervision) == manifesto["rows"] == 7410
assert [r["index"] for r in supervision] == list(range(len(inputs)))
for x, r in zip(inputs, supervision):
    esperado = {"premise", "hypothesis"} if r["task"] == "nli" else {"context", "candidate_plan"}
    assert set(x) == esperado, "Metadado misturado à entrada"
    assert r["split"] in ("train", "calibration", "test")
    assert (r["target"] in (0, 1, 2)) if r["task"] == "nli" else (0 <= r["target"] <= 1)
print("Dados:", manifesto["version"], "| exemplos:", len(inputs))
print("Entrada NLI:", list(inputs[0]))
print("Entrada probabilística:", list(next(x for x, r in zip(inputs, supervision) if r["task"] == "forecast")))

## 3. Observar entradas e distribuições antes de treinar

Na classificação, `premise` contém evidências e `hypothesis` contém a afirmação. A classe esperada fica fora da entrada.

Na previsão, `context` contém o que o observador conhece. `candidate_plan` é o nome de um campo legado que descreve **condições de continuação**; As hipóteses sobre a população de terceiros ficam na documentação do gerador, fora da entrada. O contexto vem apenas da visão autorizada e da memória do agente. A condição candidata é própria e conhecida pela implementação; não pede uma ação como saída.

A célula mostra os **cinco primeiros e cinco últimos registros de treino de cada tarefa**, com entrada completa e alvo separados. Abra os detalhes para ler os textos.
O contexto compacto tem tabelas `columns`/`rows`. `questions` define perguntas locais q1, q2 etc.; essas referências não possuem significado global. `speech` reúne falas recebidas e enviadas, sem duplicação. `facts`, `sensors`, `revelations` e `beliefs` mantêm estatutos distintos. Campos vazios são omitidos, sem significar falsidade.

O arquivo `entrada_agente.py` do repositório fornece `ObservationMemory` e `InputBuilder`: a mesma preparação é usada na massa e pode ser chamada nos hooks do agente. Ele recebe `AgentView`, não a engine. Nenhuma distribuição de implementações de terceiros, ID real, parâmetro de simulação ou justificativa do alvo faz parte da entrada.


In [ ]:
import html
from IPython.display import HTML

def mostrar_amostra(entradas, alvos, indices, titulo):
    partes = [f"<h3>{html.escape(titulo)}</h3>"]
    for i in dict.fromkeys(list(indices[:5]) + list(indices[-5:])):
        legivel = {k: json.loads(v) if k in ("premise", "hypothesis", "context") else v for k, v in entradas[i].items()}
        entrada = html.escape(json.dumps(legivel, ensure_ascii=False, indent=2))
        alvo = html.escape(json.dumps(alvos[i], ensure_ascii=False, indent=2))
        partes.append(f"<details><summary>Registro {i}</summary><b>ENTRADA</b><pre>{entrada}</pre><b>ALVO — fora da entrada</b><pre>{alvo}</pre></details>")
    display(HTML("".join(partes)))
for tarefa in ("nli", "forecast"):
    ids = [i for i, r in enumerate(supervision) if r["task"] == tarefa and r["split"] == "train"]
    alvos = [{"target": CLASSES[r["target"]] if r["task"] == "nli" else r["target"],
              "split": r["split"], "task": r["task"]} for r in supervision]
    mostrar_amostra(inputs, alvos, ids, tarefa + " — treino")

### 3.1. Classes e frequências são alvos diferentes

| Tarefa | Um alvo | Transformação da cabeça | Perda |
|---|---|---|---|
| Classificação | 0, 1 ou 2 | Softmax, três valores que somam 1 | Entropia cruzada |
| Evento | Frequência entre 0 e 1 | Sigmoid; complemento = outro desfecho | Entropia cruzada binária com alvo fracionário |

Uma frequência de `0,75` significa 72 ocorrências em 96 continuações, sob as condições daquele exemplo. Não significa uma certeza do mundo real. Cada alvo combina três continuações por mundo, em 32 mundos: 96 desfechos. Os mundos são a unidade para a incerteza, pois as três continuações compartilham o estado inicial. Os regimes têm pesos fixos iguais na geração; não são features nem identificações de outros agentes.

As três classes são equilibradas. As probabilidades **não são forçadas a 50/50**, pois isso mudaria o evento que queremos estimar.

In [ ]:
tabela = pd.DataFrame(supervision)
display(pd.crosstab(tabela["task"], tabela["split"]))
display(pd.crosstab(tabela.query("task == 'nli'")["split"], tabela.query("task == 'nli'")["target"]).rename(columns=dict(enumerate(CLASSES))))
fore = tabela.query("task == 'forecast'")
display(fore.groupby("split")["target"].agg(["count", "mean", "min", "max"]))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
tabela.query("task == 'nli'")["target"].value_counts().sort_index().rename(index=dict(enumerate(CLASSES))).plot.bar(ax=ax[0], rot=0, title="Classificação — número por classe")
for parte in ("train", "calibration", "test"):
    ax[1].hist(fore.loc[fore["split"] == parte, "target"], bins=np.linspace(0, 1, 17), alpha=.45, label=parte)
ax[1].set(title="Evento — frequências supervisionadas", xlabel="Frequência estimada", ylabel="Número de entradas")
ax[1].legend(); plt.tight_layout(); plt.show()
display(pd.crosstab(fore["round"], fore["phase"]))

## 4. Respeitar as partições

```text
Treino      → calcula média/desvio e atualiza os pesos
Calibração  → escolhe temperatura com pesos já fixos
Teste       → mede resultados depois de todas as escolhas
```

Famílias de episódios e contextos relacionados permanecem na mesma partição. O teste não é usado para escolher épocas ou hiperparâmetros. Mantemos as 400 épocas, Adam, taxa 0,01 e regularização 0,1 do piloto anterior para uma comparação controlada. Mudou a massa; a arquitetura das cabeças permanece a mesma.

Rodadas 1–6 e oito fases estão cobertas na previsão. Classificação também inclui faixas de rodadas posteriores. A massa não cobre todos os históricos ou qualquer implementação futura.

In [ ]:
from itertools import combinations
for a, b in combinations(("train", "calibration", "test"), 2):
    ga = {r["group"] for r in supervision if r["split"] == a}
    gb = {r["group"] for r in supervision if r["split"] == b}
    assert not ga & gb, "Grupo em partições diferentes"
print(json.dumps(protocol["optimization"], indent=2))
print("Épocas e hiperparâmetros fixados antes da avaliação.")

## 5. Baixar o Phi-4 e carregar o corpo congelado

**Aqui o Phi-4 é baixado.** Usamos a mesma revisão quantizada de 4 bits do piloto anterior, identificada por hashes no pacote. Esses são pesos pré-treinados do corpo; as cabeças desta execução ainda não existem.

O carregador abaixo decodifica os pesos quantizados em blocos e executa o corpo pelo PyTorch. Seu código aparece na célula para tornar o carregamento explícito. A camada de linguagem A/B/C fornece apenas um baseline da classificação; não é a cabeça treinada de outro domínio.

In [ ]:
from huggingface_hub import snapshot_download
modelo_base = json.loads((DATA_DIR / "modelo.json").read_text())
print("Baixando Phi-4:", modelo_base["repo"], "| revisão:", modelo_base["revision"])
print("Tamanho dos arquivos:", round(sum(f["bytes"] for f in modelo_base["model_files"]) / 1e9, 2), "GB")
PHI4_DIR = snapshot_download(repo_id=modelo_base["repo"], revision=modelo_base["revision"],
                            allow_patterns=[f["path"] for f in modelo_base["model_files"]])
print("Arquivos baixados em:", PHI4_DIR)

In [ ]:
"""Mesmo Phi-4 quantizado, executado por PyTorch em CUDA, MPS ou CPU.
Não usa geração, bitsandbytes, FlashAttention obrigatório, MLX ou kernel Metal próprio.
"""
from pathlib import Path
import hashlib,json,platform
import numpy as np
import transformers
import torch
from torch import nn
from torch.nn import functional as F
from safetensors import safe_open
from transformers import AutoTokenizer,Phi3Config
from transformers.models.phi3.modeling_phi3 import Phi3Model,Phi3RotaryEmbedding

NLI_SYSTEM=('Classify the relation of a hypothesis to a premise using only the premise. '
 'A = entailment: the hypothesis follows from the premise. '
 'B = neutral: the hypothesis neither follows from nor contradicts the premise. '
 'C = contradiction: the hypothesis contradicts the premise. '
 'Answer with exactly one letter: A, B, or C.')

def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def automatic_device():
 if torch.cuda.is_available():return 'cuda'
 if hasattr(torch.backends,'mps') and torch.backends.mps.is_available():return 'mps'
 return 'cpu'
def choose_device(requested='auto'):
 value=automatic_device() if requested=='auto' else requested
 if value not in ('cpu','cuda','mps'):raise ValueError('Dispositivo desconhecido')
 if value=='cuda' and not torch.cuda.is_available():raise ValueError('CUDA indisponível; use auto ou cpu')
 if value=='mps' and not torch.backends.mps.is_available():raise ValueError('MPS indisponível; use auto ou cpu')
 return value

def prepare_assets(manifest_path,model_directory=None):
 manifest=json.loads(Path(manifest_path).read_text())
 if model_directory is None:
  from huggingface_hub import snapshot_download
  model_directory=snapshot_download(repo_id=manifest['repo'],revision=manifest['revision'],allow_patterns=[r['path'] for r in manifest['model_files']])
 directory=Path(model_directory)
 for r in manifest['model_files']:
  p=directory/r['path']
  if not p.is_file() or p.stat().st_size!=r['bytes'] or sha(p)!=r['sha256']:raise ValueError(f'Peso/tokenizer ausente ou incompatível: {p}')
 return directory,manifest

def unpack(weight,scales,biases,dtype):
 """Grupos de 64; 8 nibbles por inteiro; arredondamento original fp16."""
 weight=weight.to(torch.int32)
 shifts=torch.arange(0,32,4,dtype=torch.int32,device=weight.device)
 values=((weight.unsqueeze(-1)>>shifts)&15).reshape(weight.shape[0],-1,64)
 decoded=values.float()*scales.float().unsqueeze(-1)+biases.float().unsqueeze(-1)
 return decoded.reshape(weight.shape[0],-1).to(torch.float16).to(dtype)

class PackedLinear(nn.Module):
 def __init__(self,weight,scales,biases,tile_rows=256):
  super().__init__()
  if type(tile_rows) is not int or tile_rows<1:raise ValueError('tile_rows deve ser positivo')
  self.tile_rows=tile_rows
  self.register_buffer('packed',weight.to(torch.int32));self.register_buffer('scales',scales);self.register_buffer('biases',biases)
 def forward(self,inputs):
  blocks=[]
  for start in range(0,self.packed.shape[0],self.tile_rows):
   sl=slice(start,start+self.tile_rows)
   w=unpack(self.packed[sl],self.scales[sl],self.biases[sl],inputs.dtype)
   blocks.append(F.linear(inputs,w))
  return torch.cat(blocks,dim=-1)

class PackedEmbedding(PackedLinear):
 def forward(self,ids):
  dtype=torch.float32 if ids.device.type=='cpu' else torch.float16
  shape=ids.shape;flat=ids.reshape(-1)
  # Blocagem também limita buffers de desquantização de embeddings longos.
  result=[unpack(self.packed[flat[s:s+self.tile_rows]],self.scales[flat[s:s+self.tile_rows]],self.biases[flat[s:s+self.tile_rows]],dtype) for s in range(0,len(flat),self.tile_rows)]
  return torch.cat(result,dim=0).reshape(*shape,-1)

def last_token(hidden,mask):
 lengths=mask.sum(-1)
 if bool((lengths<=0).any()):raise ValueError('Entrada vazia')
 return hidden[torch.arange(len(lengths),device=hidden.device),lengths-1]

def final_hidden(model,inputs,prefill_chunk=0):
 if not prefill_chunk:
  mask=torch.ones_like(inputs)
  return model(input_ids=inputs,attention_mask=mask,use_cache=False).last_hidden_state[:,-1]
 cache=None
 for start in range(0,inputs.shape[1],prefill_chunk):
  end=min(start+prefill_chunk,inputs.shape[1])
  mask=torch.ones((inputs.shape[0],end),dtype=inputs.dtype,device=inputs.device)
  output=model(input_ids=inputs[:,start:end],attention_mask=mask,past_key_values=cache,use_cache=True)
  cache=output.past_key_values
 return output.last_hidden_state[:,-1]

class Phi4Features:
 def __init__(self,manifest_path,model_directory=None,device='auto',tile_rows=256,prefill_chunk=None):
  self.directory,self.manifest=prepare_assets(manifest_path,model_directory)
  self.device=choose_device(device)
  self.prefill_chunk=(256 if self.device=='cpu' else 0) if prefill_chunk is None else prefill_chunk
  if type(self.prefill_chunk) is not int or self.prefill_chunk<0:raise ValueError('prefill_chunk deve ser inteiro não negativo')
  dtype=torch.float32 if self.device=='cpu' else torch.float16
  self.tokenizer=AutoTokenizer.from_pretrained(self.directory,local_files_only=True,trust_remote_code=False)
  cfg=json.loads((self.directory/'config.json').read_text())
  if cfg.get('quantization')!={'group_size':64,'bits':4}:raise ValueError('Quantização incompatível')
  cfg.pop('quantization_config',None);cfg.pop('quantization',None)
  config=Phi3Config.from_dict(cfg)
  if config.hidden_size!=5120 or config.max_position_embeddings!=16384:raise ValueError('Arquitetura incompatível')
  config._attn_implementation='sdpa';config.use_cache=False
  self.window=min(config.max_position_embeddings,self.tokenizer.model_max_length)
  with torch.device('meta'):self.model=Phi3Model(config)
  index=json.loads((self.directory/'model.safetensors.index.json').read_text())['weight_map']
  def read(name,selected=None):
   with safe_open(self.directory/index[name],framework='pt',device='cpu') as shard:
    t=shard.get_tensor(name);return t.clone() if selected is None else t[selected].clone()
  for name,module in list(self.model.named_modules()):
   if isinstance(module,(nn.Linear,nn.Embedding)):
    parent,_,child=name.rpartition('.');owner=self.model.get_submodule(parent) if parent else self.model
    values=[read(f'model.{name}.{suffix}') for suffix in ('weight','scales','biases')]
    cls=PackedEmbedding if isinstance(module,nn.Embedding) else PackedLinear
    setattr(owner,child,cls(*values,tile_rows=tile_rows).to(self.device))
  for name,_ in list(self.model.named_parameters()):
   parent,_,child=name.rpartition('.')
   setattr(self.model.get_submodule(parent),child,nn.Parameter(read('model.'+name).to(device=self.device,dtype=dtype),requires_grad=False))
  self.model.rotary_emb=Phi3RotaryEmbedding(config,device=self.device);self.model.eval();self.model.requires_grad_(False)
  letters=[self.tokenizer.encode(x,add_special_tokens=False) for x in 'ABC']
  if any(len(x)!=1 for x in letters):raise ValueError('Classes A/B/C precisam ser tokens únicos')
  ids=[x[0] for x in letters]
  self.abc_weight=unpack(*[read('lm_head.'+s,ids).to(self.device) for s in ('weight','scales','biases')],dtype)
  if any(p.device.type=='meta' for p in self.model.parameters()):raise ValueError('Peso não carregado')
  self.identity={'repo':self.manifest['repo'],'revision':self.manifest['revision'],'device':self.device,'dtype':str(dtype),'torch':torch.__version__,'transformers':transformers.__version__,'numpy':np.__version__,'platform':platform.platform(),'tile_rows':tile_rows,'prefill_chunk':self.prefill_chunk,'feature':'last nonpadding token, final hidden state','dimension':5120,'model_files':self.manifest['model_files']}
 def encode_nli(self,pair,prefix):
  if set(pair)!={'premise','hypothesis'} or any(not isinstance(v,str) for v in pair.values()):raise ValueError('Schema NLI inválido')
  value={'premise':prefix+'\n\n'+pair['premise'],'hypothesis':pair['hypothesis']}
  prompt=self.tokenizer.apply_chat_template([{'role':'system','content':NLI_SYSTEM},{'role':'user','content':json.dumps(value,ensure_ascii=False)}],tokenize=False,add_generation_prompt=True)
  return self.tokenizer.encode(prompt,add_special_tokens=False)
 @torch.inference_mode()
 def features(self,tokens):
  if not tokens or len(tokens)>self.window:raise ValueError('Contexto vazio ou acima da janela; truncamento proibido')
  inputs=torch.tensor([tokens],dtype=torch.long,device=self.device);mask=torch.ones_like(inputs)
  final=final_hidden(self.model,inputs,self.prefill_chunk)
  x=final.float().cpu().numpy()[0];abc=F.linear(final,self.abc_weight).float().cpu().numpy()[0]
  if not np.isfinite(x).all() or not np.isfinite(abc).all():raise ValueError('Features não finitas')
  return x,abc

In [ ]:
corpo = Phi4Features(DATA_DIR / "modelo.json", model_directory=PHI4_DIR)
print("Phi-4 carregado:", corpo.identity["repo"], "| dispositivo:", corpo.device)
print("Parâmetros do corpo com gradiente:", sum(p.numel() for p in corpo.model.parameters() if p.requires_grad))
assert not any(p.requires_grad for p in corpo.model.parameters())
prefixo = (DATA_DIR / "prefixo-nli.txt").read_text().rstrip("\n")
SISTEMA_EVENTO = (DATA_DIR / "system-forecast.txt").read_text().rstrip("\n")
def codificar(entrada, tarefa, prefixo_nli=prefixo):
    if tarefa == "nli":
        return corpo.encode_nli(entrada, prefixo_nli)
    mensagens = [{"role": "system", "content": SISTEMA_EVENTO},
                 {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, sort_keys=True)}]
    prompt = corpo.tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    return corpo.tokenizer.encode(prompt, add_special_tokens=False)

tokens_por_entrada = [codificar(x, r["task"]) for x, r in zip(inputs, supervision)]
assert all(0 < len(t) <= corpo.window for t in tokens_por_entrada), "Janela excedida: não truncar"
for tarefa in ("nli", "forecast"):
    comprimentos = [len(t) for t, r in zip(tokens_por_entrada, supervision) if r["task"] == tarefa]
    print(tarefa, "— tokens mín./mediana/máx.:", min(comprimentos), int(np.median(comprimentos)), max(comprimentos))

## 6. Extrair os vetores — a etapa longa

Cada entrada faz um **forward real no Phi-4**, sem gradientes. Guardamos o último vetor da sequência, com 5.120 valores. A extração não recebe o alvo.

Os checkpoints são salvos em blocos de 32 entradas. Ao repetir esta célula, blocos já salvos são reutilizados somente se o pacote e o runtime coincidirem. Uma interrupção perde no máximo o bloco em processamento. O log informa progresso e tempo observado; não existe uma duração garantida.

In [ ]:
FEATURES_DIR = WORK_DIR / "features-v02"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
identidade = {"body": corpo.identity, "data_sha256": sha(DATA_DIR / "integridade.json")}
manifesto_cache = FEATURES_DIR / "identidade.json"
if manifesto_cache.exists():
    assert json.loads(manifesto_cache.read_text()) == identidade, "Cache incompatível; use outra pasta"
else:
    manifesto_cache.write_text(json.dumps(identidade, ensure_ascii=False))
features, baseline_abc = [], []
inicio = time.perf_counter()
BLOCO = 32
for start in range(0, len(inputs), BLOCO):
    stop = min(start + BLOCO, len(inputs))
    arquivo = FEATURES_DIR / f"{start:05d}-{stop:05d}.npz"
    if arquivo.exists():
        with np.load(arquivo, allow_pickle=False) as cache:
            x, abc = cache["features"], cache["baseline_abc"]
            assert cache["indices"].tolist() == list(range(start, stop))
    else:
        pares = [corpo.features(tokens_por_entrada[i]) for i in range(start, stop)]
        x = np.stack([p[0] for p in pares]); abc = np.stack([p[1] for p in pares])
        temporario = arquivo.with_suffix(".tmp")
        with temporario.open("wb") as f:
            np.savez_compressed(f, features=x, baseline_abc=abc, indices=np.arange(start, stop))
        temporario.replace(arquivo)
    assert x.shape == (stop - start, 5120) and abc.shape == (stop - start, 3)
    assert np.isfinite(x).all() and np.isfinite(abc).all()
    features.extend(x); baseline_abc.extend(abc)
    print(f"{stop}/{len(inputs)} — {time.perf_counter() - inicio:.1f}s nesta execução", flush=True)
features = np.asarray(features, dtype=np.float32)
baseline_abc = np.asarray(baseline_abc, dtype=np.float32)
print("Matriz extraída:", features.shape)

## 7. Treinar as duas cabeças

Agora começa o **treinamento**, restrito às camadas lineares `W` e `b`. Normalizamos vetores com média e desvio do treino de cada tarefa. `backward()` calcula o gradiente, e `step()` atualiza somente a cabeça.

A classificação tem três logits; a previsão tem um logit. O alvo fracionário da previsão entra diretamente na entropia cruzada binária, sem arredondamento para uma classe. Todos os exemplos probabilísticos têm 96 continuações em 32 mundos, portanto pesos iguais nesta massa.

```text
Vetor normalizado → Wz+b → perda com o alvo → backward → Adam.step
```

A regularização L2 limita o tamanho dos pesos. Seu valor e o número de épocas são os do protocolo anterior; a célula imprime a perda ao longo do treino.

In [ ]:
# Cada cabeça é criada aqui e recebe apenas os vetores de sua partição de treino.
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.use_deterministic_algorithms(True)
cabecas = {}
for tarefa in ("nli", "forecast"):
    indices = [r["index"] for r in supervision if r["task"] == tarefa]
    meta = [supervision[i] for i in indices]
    x = features[indices]
    y = np.array([r["target"] for r in meta])
    partes = {p: np.array([j for j, r in enumerate(meta) if r["split"] == p])
              for p in ("train", "calibration", "test")}
    treino = partes["train"]
    media = x[treino].mean(axis=0).astype(np.float32)
    desvio = np.maximum(x[treino].std(axis=0), 0.001).astype(np.float32)
    z = (x - media) / desvio
    torch.manual_seed(protocol["seed"])
    camada = torch.nn.Linear(5120, 3 if tarefa == "nli" else 1)
    torch.nn.init.zeros_(camada.weight)
    torch.nn.init.zeros_(camada.bias)
    op = protocol["optimization"]
    otimizador = torch.optim.Adam(camada.parameters(), lr=op["learning_rate"])
    xt = torch.tensor(z[treino], dtype=torch.float32)
    yt = torch.tensor(y[treino], dtype=torch.long if tarefa == "nli" else torch.float32)
    historico = []
    for epoca in range(op["epochs"]):
        otimizador.zero_grad(set_to_none=True)
        logits = camada(xt)
        if tarefa == "nli":
            perda = torch.nn.functional.cross_entropy(logits, yt)
        else:
            perda = torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 0], yt)
        objetivo = perda + op["l2"] * camada.weight.square().sum() / 2
        assert torch.isfinite(objetivo), "Perda não finita"
        objetivo.backward()
        otimizador.step()
        if epoca == 0 or (epoca + 1) % 20 == 0:
            print(tarefa, "época", epoca + 1, "perda", round(perda.item(), 6))
            historico.append({"epoch": epoca + 1, "train_loss": perda.item(), "regularized_objective": objetivo.item()})
    cabecas[tarefa] = {"model": camada, "mean": media, "scale": desvio, "z": z, "y": y,
                        "parts": partes, "indices": indices, "meta": meta, "history": historico}
    print(tarefa, "— parâmetros:", sum(p.numel() for p in camada.parameters()))
    print("Histórico — início:", historico[:2], "| fim:", historico[-2:])

## 8. Calibrar e avaliar

A temperatura muda a confiança das probabilidades sem atualizar `W` e `b`. Escolhemos a temperatura pela perda na **calibração**, e só depois medimos o teste.

| Métrica | O que explica | Racional |
|---|---|---|
| Acurácia | Fração das classes corretas | Desempenho global |
| Macro F1 | Média do F1 das três classes | Evita ocultar uma classe fraca |
| Matriz de confusão | Classes reais × previstas | Mostra os tipos de erro |
| Log loss | Penaliza previsões confiantes erradas | Avalia confiança, não apenas classe |
| MSE da frequência | Distância entre previsão e frequência simulada | Mede ajuste ao alvo Monte Carlo |
| Brier do evento | Erro quadrático esperado contra o evento binário | Avalia probabilidades de desfecho |

Para frequência `q` e previsão `p`, Brier = `q(1−p)² + (1−q)p²` = `(p−q)² + q(1−q)`. Logo, MSE e Brier não são a mesma coisa. A parcela `q(1−q)` é variabilidade do evento.

Comparamos também com A/B/C do corpo congelado na classificação e com uma probabilidade constante ajustada **somente no treino** na previsão. Superar um baseline é uma evidência mais útil que olhar a métrica isolada. Linhas relacionadas por episódio não são amostras independentes; as métricas aqui são descritivas.

In [ ]:
def probabilidades(logits, tarefa, temperatura=1.0):
    logits = np.asarray(logits, dtype=np.float64) / temperatura
    if tarefa == "nli":
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)
    p = np.exp(-np.logaddexp(0, -logits[:, 0]))
    return np.column_stack([p, 1 - p])

def perda_probabilidades(p, y, tarefa):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    if tarefa == "nli":
        return float(-np.log(p[np.arange(len(y)), y.astype(int)]).mean())
    return float(-(y * np.log(p[:, 0]) + (1 - y) * np.log(p[:, 1])).mean())

def metricas_classes(previstos, esperados):
    matriz = np.zeros((3, 3), dtype=int)
    for alvo, previsto in zip(esperados, previstos):
        matriz[int(alvo), int(previsto)] += 1
    por_classe = []
    for i, classe in enumerate(CLASSES):
        tp = int(matriz[i, i])
        reais, preditos = int(matriz[i].sum()), int(matriz[:, i].sum())
        por_classe.append({"label": classe, "precision": tp / max(preditos, 1),
                          "recall": tp / max(reais, 1), "f1": 2 * tp / max(reais + preditos, 1)})
    return {"accuracy": float(np.trace(matriz) / matriz.sum()),
            "macro_f1": sum(m["f1"] for m in por_classe) / 3,
            "confusion": matriz.tolist(), "per_class": por_classe}

# Calibrar sem atualizar W ou b: selecionar T somente pela perda na calibração.
result = {}
for tarefa, info in cabecas.items():
    with torch.no_grad():
        logits = info["model"](torch.tensor(info["z"], dtype=torch.float32)).numpy()
    calibracao, teste = info["parts"]["calibration"], info["parts"]["test"]
    grade = [0.5, 0.75, 1, 1.5, 2, 3, 5, 10]
    perdas = [(perda_probabilidades(probabilidades(logits[calibracao], tarefa, t), info["y"][calibracao], tarefa), t)
              for t in grade]
    _, temperatura = min(perdas, key=lambda item: (item[0], abs(item[1] - 1)))
    info["temperature"] = temperatura
    cru = probabilidades(logits[teste], tarefa)
    calibrado = probabilidades(logits[teste], tarefa, temperatura)
    y = info["y"][teste]
    print("\n", tarefa, "— temperatura:", temperatura, "| perdas na calibração:", perdas)
    if tarefa == "nli":
        abc = probabilidades(baseline_abc[info["indices"]][teste], "nli")
        alternativas = {"raw_head": cru, "calibrated_head": calibrado, "zero_shot_phi4_abc": abc,
                        "uniform": np.full((len(teste), 3), 1 / 3)}
        result[tarefa] = {}
        for nome, p in alternativas.items():
            m = metricas_classes(p.argmax(axis=1), y)
            m["nll"] = perda_probabilidades(p, y, tarefa)
            result[tarefa][nome] = m
            print(nome, json.dumps(m, ensure_ascii=False))
    else:
        prior = float(info["y"][info["parts"]["train"]].mean())
        constante = np.tile([prior, 1 - prior], (len(teste), 1))
        # Cada alvo representa n eventos observados; ponderar pelas contagens.
        contagens = np.array([info["meta"][j]["supervision"]["trials"] for j in teste])
        result[tarefa] = {}
        for nome, p in {"raw_head": cru, "calibrated_head": calibrado, "train_prior_baseline": constante}.items():
            brier = y * (1 - p[:, 0]) ** 2 + (1 - y) * p[:, 0] ** 2
            p_seguro = np.clip(p[:, 0], 1e-12, 1 - 1e-12)
            logloss = -y * np.log(p_seguro) - (1 - y) * np.log(1 - p_seguro)
            m = {"brier_delivery": float(np.average(brier, weights=contagens)),
                 "log_loss_delivery": float(np.average(logloss, weights=contagens)),
                 "mse_monte_carlo": float(np.average((p[:, 0] - y) ** 2, weights=contagens))}
            result[tarefa][nome] = m
            print(nome, m)

## 9. Inferência com as cabeças novas

A saída da classificação é um dos três rótulos; a saída probabilística contém os dois desfechos complementares. Nenhuma saída é uma ação.

A demonstração mostra os cinco primeiros e cinco últimos exemplos de teste de cada tarefa, com suas entradas. Os vetores são os mesmos que seriam obtidos ao executar novamente o corpo congelado; reutilizá-los evita repetir essa etapa longa. O alvo é exibido para conferência e não entra no cálculo da previsão.

In [ ]:
def aplicar_cabeca(feature, pesos, tarefa):
    z = (feature - pesos["mean"]) / pesos["scale"]
    logits = z @ pesos["weight"].T + pesos["bias"]
    p = probabilidades(logits.reshape(1, -1), tarefa, float(pesos["temperature"]))[0]
    if tarefa == "nli":
        return {"classification": CLASSES[int(p.argmax())]}
    return {"entrega_no_planeta": float(p[0]), "retorno_com_sobrevivencia": float(p[1])}

def pesos_da_cabeca(info):
    return {"weight": info["model"].weight.detach().numpy(), "bias": info["model"].bias.detach().numpy(),
            "mean": info["mean"], "scale": info["scale"], "temperature": info["temperature"]}

for tarefa, info in cabecas.items():
    ids = [info["indices"][j] for j in info["parts"]["test"]]
    mostrar_amostra(inputs, [{"target": CLASSES[r["target"]] if r["task"] == "nli" else r["target"]} for r in supervision], ids, tarefa + " — inferência")
    display(pd.DataFrame([{"registro": i, **aplicar_cabeca(features[i], pesos_da_cabeca(info), tarefa)}
                          for i in dict.fromkeys(ids[:5] + ids[-5:])]))

## 10. Comparar classificação v01 e v02 com observações reproduzíveis

O teste antigo de 90 casos permanece preservado. Dois casos continham sensores ausentes da visão registrada do agente e não podem entrar numa comparação que exige somente dados autorizados. Usamos os **88 casos reproduzíveis**, com o mesmo alvo e origem, nas representações próprias de cada versão.

As entradas compactas foram reconstruídas por `InputBuilder(AgentView, catálogo público)` e seus rótulos foram conferidos. Não usamos a cabeça probabilística antiga no contrato corrigido. Os 30 casos fora do domínio antigos permanecem arquivados; seu formato em prosa precisa de um protocolo compatível antes de avaliar a cabeça nova.


In [ ]:
COMPARACAO = DATA_DIR / "comparacao-v01"
pares = [json.loads(l) for l in (COMPARACAO / "nli-benchmark-agente.jsonl").read_text().splitlines()]
assert len(pares) == 88
prefixo_v01 = (COMPARACAO / "prefixo-nli-v01.txt").read_text().rstrip("\n")
with np.load(COMPARACAO / "heads-v01/nli.npz", allow_pickle=False) as arquivo:
    pesos_v01 = {k: arquivo[k] for k in arquivo.files}
pesos_v02 = pesos_da_cabeca(cabecas["nli"])
previsoes = {"v01": [], "v02": []}
for i, caso in enumerate(pares):
    for versao, pref, pesos in (("v01", prefixo_v01, pesos_v01), ("v02", prefixo, pesos_v02)):
        tokens = codificar(caso["input_" + versao], "nli", pref)
        feature, _ = corpo.features(tokens)
        z = (feature - pesos["mean"]) / pesos["scale"]
        logits = (z @ pesos["weight"].T + pesos["bias"]).reshape(1, -1)
        previsoes[versao].append(probabilidades(logits, "nli", float(pesos["temperature"]))[0])
    if (i + 1) % 10 == 0:
        print(i + 1, "/", len(pares), flush=True)
esperado = np.array([caso["target"] for caso in pares], dtype=int)
comparacoes = {"benchmark-agente-88": {}}
for versao, probs in previsoes.items():
    p = np.asarray(probs)
    m = metricas_classes(p.argmax(axis=1), esperado)
    m["nll"] = perda_probabilidades(p, esperado, "nli")
    comparacoes["benchmark-agente-88"][versao] = m
    print(versao, json.dumps(m, ensure_ascii=False))


## 11. Salvar o treinamento e baixar os resultados

Exportamos pesos, normalização, temperatura, ordem dos rótulos, instruções e identidade do corpo. As previsões do teste mantêm grupos, rodadas e fases para análises posteriores. Resultados novos recebem a versão 02; não sobrescrevem a versão anterior.

O ZIP contém também os checkpoints de vetores, permitindo retomar o treino das cabeças sem repetir a extração. Ele pode ficar grande; baixe-o antes de encerrar a sessão.

In [ ]:
HEADS_DIR = WORK_DIR / "heads-v02"
HEADS_DIR.mkdir(exist_ok=True)
for tarefa, info in cabecas.items():
    np.savez_compressed(HEADS_DIR / (tarefa + ".npz"), **pesos_da_cabeca(info))
    ids = [info["indices"][j] for j in info["parts"]["test"]]
    registros = [{"index": i, "group": supervision[i]["group"], "round": supervision[i]["round"],
                  "phase": supervision[i]["phase"], "target": supervision[i]["target"],
                  "prediction": aplicar_cabeca(features[i], pesos_da_cabeca(info), tarefa)} for i in ids]
    (HEADS_DIR / (tarefa + "-predictions.json")).write_text(json.dumps(registros, ensure_ascii=False, indent=2))
artefato = {"version": "mission-pilot-heads-v2", "dataset": manifesto["version"],
            "window_tokens": corpo.window, "body_identity": corpo.identity,
            "data_sha256": sha(DATA_DIR / "integridade.json"), "nli_labels": CLASSES,
            "probability_output_keys": ["entrega_no_planeta", "retorno_com_sobrevivencia"],
            "heads": {t: sha(HEADS_DIR / (t + ".npz")) for t in cabecas},
            "input_contract": "agent-view-compact; NLI premise/hypothesis; forecast context/candidate_plan",
            "truncation": False}
(HEADS_DIR / "artefato.json").write_text(json.dumps(artefato, ensure_ascii=False, indent=2))
resultado = {"tasks": result, "common_benchmarks": comparacoes,
             "protocol": protocol, "training_history": {t: c["history"] for t, c in cabecas.items()}}
(WORK_DIR / "resultado-v02.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2))
saida = WORK_DIR.parent / "treinamento-aula05-v02-resultados.zip"
with zipfile.ZipFile(saida, "w", zipfile.ZIP_DEFLATED) as zipado:
    for path in WORK_DIR.rglob("*"):
        if path.is_file():
            zipado.write(path, path.relative_to(WORK_DIR))
print("Salvo:", saida)
files.download(str(saida))

## 12. Interpretar a evidência

1. Quais parâmetros tiveram gradientes? Onde aparece sua atualização?
2. Qual é a diferença entre classe, confiança na classe e probabilidade de um evento?
3. A cabeça probabilística superou a previsão constante no teste?
4. Calibrar ajudou a confiança? Alterou a classe mais provável?
5. A versão 02 melhorou nos mesmos benchmarks NLI da versão 01?
6. Quais conclusões ficam limitadas pelos 32 mundos pareados, pela dependência entre episódios e pelas rodadas cobertas?

Um resultado favorável no teste não demonstra cobertura de todas as estratégias. O modelo continua sendo um classificador de evidências e estimador condicionado de probabilidades; a decisão de ação pertence à implementação do agente.